In [2]:
using CovidSim_ilm

┌ Info: Precompiling CovidSim_ilm [top-level]
└ @ Base loading.jl:1423


In [3]:
cs = CovidSim_ilm

CovidSim_ilm

In [4]:
using StatsBase
using TypedTables
using BenchmarkTools
using Distributions
using YAML
using PrettyPrint
using LinearAlgebra
using Dates

In [5]:
cd(joinpath(homedir(),"Dropbox/Covid Modeling/Covid-ILM/source"))

# Test model setup

In [80]:
ndays = 180
locale = 38015
model180 = buildsim(ndays, locale;  
    day1 = Date("2020-01-01", "yyyy-mm-dd"),
    dovax = true,
    paramdir = "../parameters",
    geofilename = "../data/geo2data.csv", 
    socialfilename = "socialparams.yml",
    vaccinefilename = "vaccines.yml",
    variantfilename = "variants.yml",
);

In [81]:
keys(model180)


# ### Social Parameters

(:ndays, :day1, :locales, :dat, :series, :geo, :transitionset, :vaxset, :vaxschedset, :infectset, :social, :trvec)

In [82]:
fieldnames(typeof(model180.social))

(:gammashape, :contactfactors, :touchfactors)

In [83]:
typeof(model180.social.gammashape)

Float64

In [84]:
model180.social.contactfactors 

4×5 Matrix{Float64}:
 1.1  2.1  2.1  1.7  1.0
 1.1  2.0  2.0  1.6  0.9
 0.7  1.0  1.0  0.7  0.6
 0.5  0.6  0.6  0.5  0.5

In [85]:
model180.social.contactfactors[Int(sick)-4, Int(age0_19)]

0.7

In [86]:
touchfactors = model180.social.touchfactors

6×5 Matrix{Float64}:
 0.55  0.63  0.61  0.41  0.35
 0.55  0.63  0.61  0.41  0.35
 0.55  0.63  0.61  0.41  0.35
 0.55  0.62  0.58  0.41  0.28
 0.28  0.35  0.3   0.18  0.18
 0.18  0.18  0.18  0.18  0.18

## Infectset

In [87]:
model180.infectset

OrderedCollections.LittleDict{Symbol, Infectparams, Vector{Symbol}, Vector{Infectparams}} with 5 entries:
  :alpha       => Infectparams([0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.…
  :omicron_ba2 => Infectparams([0.0, 0.545, 1.175, 1.325, 1.475, 1.475, 1.325, …
  :omicron_ba1 => Infectparams([0.0, 0.445, 0.975, 1.125, 1.275, 1.275, 1.125, …
  :base        => Infectparams([0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.…
  :delta       => Infectparams([0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.…

### Parameters for infection based on variant

In [88]:
fieldnames(typeof(model180.infectset[:omicron_ba1]))

(:sendrisk, :recvrisk, :recovery_immunity, :immunehalflife, :basemultiplier)

In [89]:
println(CovidSim_ilm.variantlist)

[:alpha, :omicron_ba2, :omicron_ba1, :base, :delta]


In [90]:
pprintln(keys(model180.infectset))

{:alpha, :omicron_ba2, :omicron_ba1, 
 :base, :delta}


In [91]:
pprintln(model180.infectset[:base])

Infectparams(
  sendrisk=[0.0, 0.3, 0.65, 0.75, 
            0.85, 0.85, 0.75, 0.7, 
            0.65, 0.6, 0.5, 0.2, 
            0.1, 0.1, 0.1, 0.05, 
            0.05, 0.0, 0.0, 0.0, 
            0.0, 0.0, 0.0, 0.0, 0.0],
  recvrisk=[0.1, 0.39, 0.44, 0.54, 
            0.56],
  recovery_immunity={:alpha : 0.8, 
                     :omicron_ba2 : 0.5, 
                     :omicron_ba1 : 0.5, 
                     :base : 0.8, 
                     :delta : 0.7},
  immunehalflife=360,
  basemultiplier=1.0,
)


In [92]:
sendbase = model180.infectset[:base].sendrisk
recvbase = model180.infectset[:base].recvrisk

5-element Vector{Float64}:
 0.1
 0.39
 0.44
 0.54
 0.56

In [93]:
pprint(model180.infectset[:alpha])

Infectparams(
  sendrisk=[0.0, 0.3, 0.65, 0.75, 
            0.85, 0.85, 0.75, 0.7, 
            0.65, 0.6, 0.5, 0.2, 
            0.1, 0.1, 0.1, 0.05, 
            0.05, 0.0, 0.0, 0.0, 
            0.0, 0.0, 0.0, 0.0, 0.0],
  recvrisk=[0.12, 0.46799999999999997, 
            0.528, 0.648, 0.672],
  recovery_immunity={:alpha : 0.8, 
                     :omicron_ba2 : 0.6, 
                     :omicron_ba1 : 0.6, 
                     :base : 0.8, 
                     :delta : 0.7},
  immunehalflife=360,
  basemultiplier=1.2,
)

## Transition parameters

In [94]:
model180.transitionset # the decision transition matrices for all age groups are loaded

Dict{Symbol, CovidSim_ilm.Transitionparams} with 5 entries:
  :alpha       => Transitionparams(nothing, Transitionfactors(nothing, nothing))
  :omicron_ba2 => Transitionparams(nothing, Transitionfactors(nothing, nothing))
  :omicron_ba1 => Transitionparams(nothing, Transitionfactors(nothing, nothing))
  :base        => Transitionparams(Agetree(Dict(5=>[0.0 0.4 … 0.0 0.0; 0.0 0.3 …
  :delta       => Transitionparams(nothing, Transitionfactors(nothing, nothing))

In [95]:
basetransition = model180.transitionset[:base]
println("Types...")
println(typeof(model180.transitionset))
println(typeof(basetransition))
println(typeof(basetransition.tree))
println(typeof(basetransition.tree.age0_19))
println(typeof(basetransition.tree.age0_19[5]))
println()
pprintln(basetransition)

Types...
Dict{Symbol, CovidSim_ilm.Transitionparams}
CovidSim_ilm.Transitionparams
CovidSim_ilm.Agetree
Dict{Int64, Matrix{Float64}}
Matrix{Float64}

CovidSim_ilm.Transitionparams(
  tree=CovidSim_ilm.Agetree(
         age0_19={
                   5 : Matrix{Float64}(),
                   25 : Matrix{Float64}(),
                   9 : Matrix{Float64}(),
                   14 : Matrix{Float64}(),
                   19 : Matrix{Float64}(),
                 },
         age20_39={
                    5 : Matrix{Float64}(),
                    25 : Matrix{Float64}(),
                    9 : Matrix{Float64}(),
                    14 : Matrix{Float64}(),
                    19 : Matrix{Float64}(),
                  },
         age40_59={
                    5 : Matrix{Float64}(),
                    25 : Matrix{Float64}(),
                    9 : Matrix{Float64}(),
                    14 : Matrix{Float64}(),
                    19 : Matrix{Float64}(),
                  },
         age60_79={


In [96]:
fieldnames(typeof(basetransition.tree))

(:age0_19, :age20_39, :age40_59, :age60_79, :age80_up)

In [97]:
keys(basetransition.tree.age0_19)

KeySet for a Dict{Int64, Matrix{Float64}} with 5 entries. Keys:
  5
  25
  9
  14
  19

In [98]:
age0_19tree = basetransition.tree.age0_19

Dict{Int64, Matrix{Float64}} with 5 entries:
  5  => [0.0 0.4 … 0.0 0.0; 0.0 0.3 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 1.0…
  25 => [1.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0; 0.976 0.0 … 0.0 0.024; 0.91 0.0 …
  9  => [0.9 0.0 … 0.0 0.0; 0.4 0.0 … 0.0 0.0; 0.0 0.0 … 0.05 0.0; 0.0 0.0 … 1.…
  14 => [1.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0; 0.85 0.0 … 0.03 0.0; 0.692 0.0 ……
  19 => [1.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.891 0.0 … 0…

In [99]:
age0_19tree[5]

4×6 Matrix{Float64}:
 0.0  0.4  0.5  0.1  0.0  0.0
 0.0  0.3  0.6  0.1  0.0  0.0
 0.0  0.0  0.0  1.0  0.0  0.0
 0.0  0.0  0.0  0.0  1.0  0.0

In [100]:
basetransition.factors

CovidSim_ilm.Transitionfactors(nothing, nothing)

In [101]:
basetransition.factors.vaxhalflifeadjust

## Design a new sanity check

### Check rows

In [102]:
anywrong = false
for checkpt in [5,9,14,19,25]
    for row in 1:4
        if isapprox(sum(age0_19tree[checkpt][row,:]), 1.0)
        else
            println("row probabilities not equal 1.0: $checkpoint, $row: $(age0_19[checkpt][row,:])")
            anywrong = true
        end
    end
end
anywrong || println("All probability rows sum to 1.0")

All probability rows sum to 1.0


### Check outcomes are all dead or recovered

In [324]:
transitionset = model180.transitionset
variant = :base
setage = age0_19
breakday = 5
checkpoints = [5,9,14,19,25]
agetrees = transitionset[variant].tree

fieldnames(typeof(agetrees))

(:age0_19, :age20_39, :age40_59, :age60_79, :age80_up)

In [379]:
thisage = age80_up
for checkday in [5,9,14,19,25]
    println("******* checkday $checkday")
    display(getfield(agetrees, Symbol(thisage))[checkday])
    println()
end

******* checkday 5


4×6 Matrix{Float64}:
 0.0  0.1  0.5  0.4  0.0  0.0
 0.0  0.1  0.5  0.4  0.0  0.0
 0.0  0.1  0.5  0.4  0.0  0.0
 0.0  0.0  0.0  0.4  0.6  0.0


******* checkday 9


4×6 Matrix{Float64}:
 0.5  0.0  0.0  0.5  0.0  0.0
 0.0  0.0  0.4  0.6  0.0  0.0
 0.0  0.0  0.0  0.6  0.4  0.0
 0.0  0.0  0.0  0.0  1.0  0.0


******* checkday 14


4×6 Matrix{Float64}:
 0.7   0.0  0.3  0.0  0.0   0.0
 0.7   0.0  0.0  0.3  0.0   0.0
 0.7   0.0  0.0  0.1  0.2   0.0
 0.12  0.0  0.0  0.0  0.67  0.21


******* checkday 19


4×6 Matrix{Float64}:
 1.0   0.0  0.0  0.0  0.0   0.0
 1.0   0.0  0.0  0.0  0.0   0.0
 0.0   0.0  0.0  0.0  1.0   0.0
 0.49  0.0  0.0  0.0  0.24  0.27


******* checkday 25


4×6 Matrix{Float64}:
 1.0    0.0  0.0  0.0  0.0  0.0
 1.0    0.0  0.0  0.0  0.0  0.0
 0.682  0.0  0.0  0.0  0.0  0.318
 0.676  0.0  0.0  0.0  0.0  0.324

In [337]:
torecovered = Dict(k => 0.0 for k in [nil, mild, sick, severe])
todead = Dict(k => 0.0 for k in [nil, mild, sick, severe])

Dict{condition, Float64} with 4 entries:
  mild   => 0.0
  severe => 0.0
  nil    => 0.0
  sick   => 0.0

In [338]:
currage = age0_19
thisagetree = getfield(agetrees, Symbol(currage))
res = copy(thisagetree[5])
firstpass = true
for checkpt in checkpoints[2:end]
    println("*** $checkpt")
    # torecovered[nil] += sum(view(res, :, 1))
    # todead[nil] += sum(view(res, :, 6))
    # display(res)
    

    trconds = transpose(copy(res[:, 2:5]))
    if firstpass
        for i in 1
            res[:] = trconds[:, i] .* thisagetree[checkpt]
            torecovered[nil] += sum(res[:, 1])
            todead[nil] += sum(res[:, 6])
            # trconds = transpose(copy(res[:, 2:5]))
        end
        firstpass = false
    else
        for i in 1:4
            res[:] = trconds[:, i] .* thisagetree[checkpt]
            torecovered[nil] += sum(res[:, 1])
            todead[nil] += sum(res[:, 6])
            # trconds = transpose(copy(res[:, 2:5]))
        end
    end

    println("recovered ", torecovered[nil], " dead ", todead[nil], " all ", torecovered[nil]+todead[nil])
end


*** 9
recovered 0.56 dead 0.0 all 0.56
*** 14
recovered 0.9782100000000001 dead 3.0000000000000008e-5 all 0.9782400000000001
*** 19
recovered 0.9782100000000001 dead 3.0000000000000008e-5 all 0.9782400000000001
*** 25
recovered 0.9782100000000001 dead 3.0000000000000008e-5 all 0.9782400000000001


In [336]:
thisagetree[25]

4×6 Matrix{Float64}:
 1.0    0.0  0.0  0.0  0.0  0.0
 1.0    0.0  0.0  0.0  0.0  0.0
 0.76   0.0  0.0  0.0  0.0  0.24
 0.688  0.0  0.0  0.0  0.0  0.312

In [260]:
res

4×6 Matrix{Float64}:
 0.0        0.0  0.0  0.0  0.0  0.0
 0.0        0.0  0.0  0.0  0.0  0.0
 0.0        0.0  0.0  0.0  0.0  0.0
 0.0291309  0.0  0.0  0.0  0.0  0.00288108

In [261]:
trconds

4×4 transpose(::Matrix{Float64}) with eltype Float64:
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.032012

In [51]:
age0_19tree[9]

4×6 Matrix{Float64}:
 0.9  0.0  0.0  0.1   0.0   0.0
 0.4  0.0  0.6  0.0   0.0   0.0
 0.0  0.0  0.0  0.95  0.05  0.0
 0.0  0.0  0.0  0.0   1.0   0.0

In [53]:
trconds[:,1] .* age0_19tree[9]

4×6 Matrix{Float64}:
 0.36  0.0  0.0  0.04   0.0    0.0
 0.2   0.0  0.3  0.0    0.0    0.0
 0.0   0.0  0.0  0.095  0.005  0.0
 0.0   0.0  0.0  0.0    0.0    0.0

In [54]:
trconds[:,2] .* age0_19tree[9]

4×6 Matrix{Float64}:
 0.27  0.0  0.0   0.03   0.0    0.0
 0.24  0.0  0.36  0.0    0.0    0.0
 0.0   0.0  0.0   0.095  0.005  0.0
 0.0   0.0  0.0   0.0    0.0    0.0

In [ ]:
currage = age0_19
thisagetree = getfield(agetrees, Symbol(currage))

In [398]:
cs.sanitycheck(agetrees)

for agegroup age0_19
    starting condition nil
        recovered => 0.9902129199999999
        dead => 0.00978708
    OK  Prob total: 0.9999999999999999
    starting condition mild
        recovered => 0.9902129199999999
        dead => 0.00978708
    OK  Prob total: 0.9999999999999999
    starting condition sick
        recovered => 0.9902129199999999
        dead => 0.00978708
    OK  Prob total: 0.9999999999999999
    starting condition severe
        recovered => 0.9902129199999999
        dead => 0.00978708
    OK  Prob total: 0.9999999999999999
for agegroup age20_39
    starting condition nil
        recovered => 0.9970437952
        dead => 0.0029562048
    OK  Prob total: 0.9999999999999999
    starting condition mild
        recovered => 0.9970437952
        dead => 0.0029562048
    OK  Prob total: 0.9999999999999999
    starting condition sick
        recovered => 0.9970437952
        dead => 0.0029562048
    OK  Prob total: 0.9999999999999999
    starting condition severe
 